#Подход к решению

События очищены от полных дубликатов, платформы приведены к единому виду, для признаков оставлены только события внутри суточного окна, отсортированные по времени. По истории событий куки я построила поведенческие признаки: паузы между событиями и их разброс, энтропию ритма, доли типов событий, разнообразие объявлений, категорий и запросов, глубину выдачи, флаги из разобранного User-Agent, координаты курсора и возраст куки. Сильнее всего помогли признаки по объявлениям: сколько других кук смотрели объявление за последние 7 дней и сглаженная доля ботов среди его прошлых посетителей, посчитанная только по меткам строго предыдущих дней, чтобы не было утечки. Валидация временная, как и соотношение train и test: три фолда, обучение на ранних днях и проверка на следующих трёх, метрика из официального metric.py. По сравнению с baseline из quickstart (0.095) поведенческие признаки с LightGBM дали 0.755, признаки по объявлениям подняли результат до 0.903 на валидации, а на скрытом тесте вышло 0.886. Главное ограничение в том, что признаки по объявлениям работают, пока боты ходят по пересекающимся объявлениям, а без них качество падает примерно до 0.75

In [1]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

from metric import precision_at_recall


RANDOM_STATE = 42

In [2]:
train = pd.read_csv('data/train.csv', parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
test = pd.read_csv('data/test.csv', parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
events = pd.read_csv('data/events.csv.gz', parse_dates=['event_ts'])

#смотрим размеры и проверяем, что тест идёт по времени после train, от этого зависит выбор валидации
print(train.shape, test.shape, events.shape)
print('доля ботов в train:', round(train['target'].mean(), 4))
print('train:', train['window_start_ts'].min(), '-', train['window_start_ts'].max())
print('test:', test['window_start_ts'].min(), '-', test['window_start_ts'].max())

(11091, 5) (4909, 4) (328905, 14)
доля ботов в train: 0.0811
train: 2026-04-06 00:00:00 - 2026-04-19 00:00:00
test: 2026-04-20 00:00:00 - 2026-04-26 00:00:00


In [3]:
#смотрим, какие значения бывают в platform, где пропуски и есть ли дубликаты
print(events['platform'].value_counts())
print()
print(events.isna().mean().round(3))
print()
print('полных дубликатов событий:', events.duplicated().sum())
print('дубликатов cookie_id в train/test:', train['cookie_id'].duplicated().sum(), test['cookie_id'].duplicated().sum())

platform
desktop    46866
WEB        46476
Web        46365
web        45951
ANDROID    42462
Android    42254
android    42159
iphone      4103
IOS         4100
iOS         4091
ios         4078
Name: count, dtype: int64

cookie_id        0.000
event_ts         0.000
eid              0.000
event_name       0.000
platform         0.000
user_agent       0.000
item_id          0.348
item_category    0.100
item_location    0.072
seller_type      0.407
search_query     0.695
search_page      0.695
pointer_x        0.670
pointer_y        0.670
dtype: float64

полных дубликатов событий: 4863
дубликатов cookie_id в train/test: 0 0


In [4]:
#полные дубликаты событий удаляем, у ботов и людей их доля одинаковая, так что это просто шум
events = events.drop_duplicates()
#одна и та же платформа записана по-разному (WEB/Web/web, iphone/iOS), приводим к одному виду
events['platform'] = events['platform'].str.lower()
events['platform'] = events['platform'].replace({'desktop': 'web', 'iphone': 'ios'})

print(events['platform'].value_counts())

platform
web        182943
android    124988
ios         16111
Name: count, dtype: int64


In [5]:
#объединяем train и test, чтобы признаки для обоих считались одним кодом
meta = pd.concat([train, test], ignore_index=True)

ev = events.merge(meta[['cookie_id', 'window_start_ts', 'window_end_ts']], on='cookie_id')
print('всего событий:', len(ev))
#берём только события внутри окна, около 12% событий лежат после его конца
ev = ev[ev['event_ts'] >= ev['window_start_ts']]
ev = ev[ev['event_ts'] < ev['window_end_ts']]
print('событий внутри окна:', len(ev))

#в файле события перемешаны, без сортировки паузы между событиями считаются неправильно
ev = ev.sort_values(['cookie_id', 'event_ts', 'eid'])
ev = ev.reset_index(drop=True)

#пауза в секундах от предыдущего события этой же куки, у первого события NaN
ev['dt'] = ev.groupby('cookie_id')['event_ts'].diff().dt.total_seconds()
ev['hour'] = ev['event_ts'].dt.hour

#номер дня от начала train, нужен для временной валидации и признаков по объявлениям
first_day = train['window_start_ts'].min()
ev['day'] = (ev['window_start_ts'] - first_day).dt.days
train['day'] = (train['window_start_ts'] - first_day).dt.days

всего событий: 324042
событий внутри окна: 283833


In [6]:
#сравниваем ботов и людей в train, отсюда выбирала признаки и пороги
bots = ev[ev['cookie_id'].isin(train.loc[train['target'] == 1, 'cookie_id'])]
humans = ev[ev['cookie_id'].isin(train.loc[train['target'] == 0, 'cookie_id'])]

print('медиана паузы между событиями, сек: боты', bots['dt'].median(), 'люди', humans['dt'].median())
print('доля пауз < 11 сек: боты', round((bots['dt'] < 11).mean(), 3), 'люди', round((humans['dt'] < 11).mean(), 3))
print('доля событий 0-5 часов: боты', round((bots['hour'] < 6).mean(), 3), 'люди', round((humans['hour'] < 6).mean(), 3))
print('pointer_x 90-й перцентиль: боты', bots['pointer_x'].quantile(0.9), 'люди', humans['pointer_x'].quantile(0.9))
print()
print(pd.concat([bots['event_name'].value_counts(normalize=True), humans['event_name'].value_counts(normalize=True)], axis=1, keys=['bots', 'humans']).round(3))

медиана паузы между событиями, сек: боты 14.0 люди 47.0
доля пауз < 11 сек: боты 0.382 люди 0.12
доля событий 0-5 часов: боты 0.243 люди 0.13
pointer_x 90-й перцентиль: боты 1164.5 люди 1726.0

                       bots  humans
event_name                         
item_view             0.437   0.365
search_results_view   0.352   0.305
photo_swipe           0.070   0.123
seller_page_view      0.059   0.053
favorite_add          0.027   0.066
contact_phone_show    0.026   0.037
contact_chat_open     0.014   0.020
login                 0.008   0.022
contact_message_sent  0.008   0.010


In [7]:
#энтропия распределения значений: чем она меньше, тем однообразнее поведение
def entropy(values):
    probs = values.value_counts(normalize=True)
    return -(probs * np.log(probs)).sum()


def time_features(ev):
    df = ev[['cookie_id', 'event_ts', 'dt', 'hour']].copy()
    df['fast'] = df['dt'] < 11 #порог 11 сек подобрала по данным: у ботов 38% пауз короче, у людей 12%
    df['new_session'] = df['dt'].isna() | (df['dt'] > 1500) #новая сессия, если перерыв больше 25 минут
    df['night'] = df['hour'] < 6 #с 0 до 6 часов боты активнее людей почти в 2 раза
    #паузы бинним по логарифму, чтобы энтропия показывала, насколько однообразный ритм у куки
    df['dt_bin'] = np.floor(np.log1p(df['dt']))

    g = df.groupby('cookie_id')
    res = pd.DataFrame(index=g.size().index)
    res['n_events'] = g.size()
    res['span'] = (g['event_ts'].max() - g['event_ts'].min()).dt.total_seconds()
    res['dt_min'] = g['dt'].min()
    res['dt_q25'] = g['dt'].quantile(0.25)
    res['dt_median'] = g['dt'].median()
    res['dt_q75'] = g['dt'].quantile(0.75)
    res['dt_mean'] = g['dt'].mean()
    res['dt_std'] = g['dt'].std()
    res['dt_cv'] = res['dt_std'] / (res['dt_mean'] + 1) #коэффициент вариации пауз, +1 чтобы не делить на 0
    res['dt_entropy'] = g['dt_bin'].agg(entropy)
    res['fast_share'] = g['fast'].mean()
    res['n_sessions'] = g['new_session'].sum()
    res['events_per_session'] = res['n_events'] / res['n_sessions']
    res['night_share'] = g['night'].mean()
    res['n_hours'] = g['hour'].nunique()
    res['hour_entropy'] = g['hour'].agg(entropy)
    return res

In [8]:
def event_features(ev):
    counts = pd.crosstab(ev['cookie_id'], ev['event_name']) #сколько раз и какую долю от всех событий куки занимает каждый тип события
    shares = counts.div(counts.sum(axis=1), axis=0)
    counts.columns = ['cnt_' + c for c in counts.columns]
    shares.columns = ['share_' + c for c in shares.columns]
    res = counts.join(shares)

    df = ev[['cookie_id', 'event_name', 'dt']].copy()
    df['prev_event'] = df.groupby('cookie_id')['event_name'].shift(1) #смотрим соседние события, чтобы понять, откуда пришли на объявление и что сделали дальше
    df['next_event'] = df.groupby('cookie_id')['event_name'].shift(-1)

    views = df[df['event_name'] == 'item_view'].copy()
    views['with_photo'] = views['next_event'] == 'photo_swipe' #человек обычно листает фото после открытия объявления, парсеру это не нужно
    views['from_search'] = views['prev_event'] == 'search_results_view'

    g = views.groupby('cookie_id')
    res['view_with_photo'] = g['with_photo'].mean()
    res['view_from_search'] = g['from_search'].mean()
    res['view_dt_median'] = g['dt'].median()
    return res

In [9]:
def content_features(ev):
    df = ev[['cookie_id', 'item_id', 'item_category', 'item_location', 'seller_type', 'search_query', 'search_page']].copy()
    df['is_pro'] = df['seller_type'] == 'pro'

    g = df.groupby('cookie_id')
    res = pd.DataFrame(index=g.size().index)
    res['n_items'] = g['item_id'].nunique()
    res['items_ratio'] = res['n_items'] / (g['item_id'].count() + 1) #доля уникальных объявлений среди просмотров, +1 для кук без объявлений
    res['n_categories'] = g['item_category'].nunique()
    res['n_locations'] = g['item_location'].nunique()
    res['pro_share'] = g['is_pro'].mean()
    res['n_queries'] = g['search_query'].nunique()
    res['queries_ratio'] = res['n_queries'] / (g['search_query'].count() + 1)
    res['page_max'] = g['search_page'].max() #боты листают выдачу глубже людей
    res['page_mean'] = g['search_page'].mean()
    return res

In [10]:
def ua_features(ev):
    df = ev[['cookie_id', 'platform', 'user_agent']].copy()
    ua = df['user_agent'] #строк UA 148 штук, поэтому целиком не кодируем, а достаём из них флаги
    df['is_headless'] = ua.str.contains('HeadlessChrome')
    df['is_script'] = ua.str.contains('curl|Scrapy|python|node-fetch|Go-http-client')
    df['is_app'] = ua.str.startswith('Avito')
    df['is_firefox'] = ua.str.contains('Firefox')
    df['is_yandex'] = ua.str.contains('YaBrowser')
    df['is_linux'] = ua.str.contains('X11; Linux')
    df['chrome_version'] = ua.str.extract(r'Chrome/(\d+)')[0].astype(float) #версия Chrome числом, у не-Chrome браузеров NaN
    df['ua_mobile'] = ua.str.contains('Android|iPhone')
    #мобильный UA на web или десктопный UA в приложении
    df['mismatch'] = df['ua_mobile'] == (df['platform'] == 'web')
    df['is_web'] = df['platform'] == 'web'
    df['is_android'] = df['platform'] == 'android'
    df['is_ios'] = df['platform'] == 'ios'

    g = df.groupby('cookie_id')
    res = pd.DataFrame(index=g.size().index)
    #для флагов берём долю событий куки, где флаг выполняется
    for col in ['is_headless', 'is_script', 'is_app', 'is_firefox', 'is_yandex', 'is_linux', 'mismatch', 'is_web', 'is_android', 'is_ios']:
        res[col] = g[col].mean()
    res['chrome_version'] = g['chrome_version'].max()
    res['n_user_agents'] = g['user_agent'].nunique()
    res['n_platforms'] = g['platform'].nunique()
    return res

In [11]:
def pointer_features(ev):
    web = ev[ev['platform'] == 'web'].copy() #координаты курсора есть только на web, в приложениях их нет, поэтому считаем только по web
    web['no_pointer'] = web['pointer_x'].isna() #у ботов на web курсор часто вообще не передаётся

    g = web.groupby('cookie_id')
    res = pd.DataFrame(index=g.size().index)
    res['no_pointer_share'] = g['no_pointer'].mean()
    res['x_max'] = g['pointer_x'].max()
    res['y_max'] = g['pointer_y'].max()
    res['x_mean'] = g['pointer_x'].mean()
    res['y_mean'] = g['pointer_y'].mean()
    res['x_std'] = g['pointer_x'].std()
    res['y_std'] = g['pointer_y'].std()
    return res

In [12]:
#сервисы сбора данных ходят по одним и тем же объявлениям, поэтому смотрим, насколько популярны объявления куки
items = ev.loc[ev['item_id'].notna(), ['cookie_id', 'item_id', 'day']].drop_duplicates()
item_day_counts = items.groupby(['item_id', 'day']).size().reset_index(name='n_cookies')

popularity_parts = []
#для каждого дня считаем, сколько кук смотрели объявление за последние 7 дней, только прошлое и текущий день
for day in sorted(items['day'].unique()):
    recent = item_day_counts[(item_day_counts['day'] <= day) & (item_day_counts['day'] > day - 7)]
    recent_popularity = recent.groupby('item_id')['n_cookies'].sum()
    today = items[items['day'] == day].copy()
    #минус 1 чтобы не считать саму куку
    today['popularity'] = today['item_id'].map(recent_popularity) - 1
    popularity_parts.append(today)

popularity = pd.concat(popularity_parts)
popularity_features = popularity.groupby('cookie_id')['popularity'].agg(['mean', 'max'])
popularity_features.columns = ['item_pop_mean', 'item_pop_max']

print(popularity_features.join(train.set_index('cookie_id')['target'], how='inner').groupby('target').median())

        item_pop_mean  item_pop_max
target                             
0            0.714286           2.0
1            1.222222           3.0


In [13]:
features = time_features(ev)
features = features.join(event_features(ev))
features = features.join(content_features(ev))
features = features.join(ua_features(ev))
features = features.join(pointer_features(ev))
features = features.join(popularity_features)

features = features.reindex(meta['cookie_id']) #выравниваем порядок строк по cookie_id из train и test
#сколько дней кука существовала к началу окна, у ботов куки обычно свежее
features['cookie_age_days'] = (meta['window_start_ts'] - meta['cookie_created_at']).dt.total_seconds().values / 86400

print(features.shape)
print('кук без событий в окне:', features['n_events'].isna().sum()) #проверяем, что у каждой куки есть хотя бы одно событие в окне

(16000, 69)
кук без событий в окне: 0


In [14]:
#доля ботов среди прошлых посетителей объявления, аналог target encoding, но строго по прошлому
def item_bot_rate(items, labels, cutoff_day, alpha):
    labeled = items.merge(labels, on='cookie_id')
    labeled = labeled[labeled['day'] < cutoff_day] #метки берём только до cutoff_day, в валидации это граница фолда
    prior = labeled['target'].mean()

    parts = []
    for day in sorted(items['day'].unique()):
        #доля ботов только по размеченным кукам из прошлых дней, иначе утечка своей же метки
        past = labeled[labeled['day'] < day]
        stats = past.groupby('item_id')['target'].agg(['sum', 'count'])
        today = items[items['day'] == day].merge(stats, left_on='item_id', right_index=True, how='left')
        today['sum'] = today['sum'].fillna(0)
        today['count'] = today['count'].fillna(0)
        #сглаживаем к общей доле ботов, чтобы редкие объявления не давали 0 или 1
        today['bot_rate'] = (today['sum'] + alpha * prior) / (today['count'] + alpha)
        parts.append(today)

    rates = pd.concat(parts)
    res = rates.groupby('cookie_id')['bot_rate'].agg(['mean', 'max']) #среднее и максимум по всем объявлениям куки
    res.columns = ['item_bot_rate_mean', 'item_bot_rate_max']
    return res

In [15]:
X_train = features.loc[train['cookie_id']]
y_train = train['target'].values

folds = []
#тест идёт позже train по времени, поэтому валидируемся тоже по времени, а не случайным сплитом
for start in [7, 9, 11]:
    fit_mask = (train['day'] < start).values
    valid_mask = ((train['day'] >= start) & (train['day'] < start + 3)).values
    folds.append((start, fit_mask, valid_mask))
    print('fold', start, 'train:', fit_mask.sum(), 'valid:', valid_mask.sum(), 'ботов в valid:', y_train[valid_mask].sum())

fold 7 train: 5930 valid: 2470 ботов в valid: 188
fold 9 train: 7599 valid: 2254 ботов в valid: 191
fold 11 train: 9140 valid: 1951 ботов в valid: 160


In [16]:
lgb_params = { #маленький learning rate и много деревьев стабильнее на валидации, 15 листьев подобрала из 7, 15 и 31
    'n_estimators': 1000,
    'learning_rate': 0.01,
    'num_leaves': 15,
    'min_child_samples': 30,
    'subsample': 0.8,
    'subsample_freq': 1,
    'colsample_bytree': 0.7,
    'random_state': RANDOM_STATE,
    'verbose': -1,
}

#обучаем на каждом фолде и считаем официальную метрику, ROC-AUC для диагностики
def run_cv(make_model, columns, use_bot_rate):
    scores = []
    aucs = []
    for start, fit_mask, valid_mask in folds:
        X = X_train[columns]
        if use_bot_rate:
            #доля ботов пересчитывается в каждом фолде только по меткам обучающей части
            rates = item_bot_rate(items, train[['cookie_id', 'target']], start, 4)
            X = X.join(rates)
        model = make_model()
        model.fit(X[fit_mask], y_train[fit_mask])
        pred = model.predict_proba(X[valid_mask])[:, 1]
        scores.append(precision_at_recall(y_train[valid_mask], pred))
        aucs.append(roc_auc_score(y_train[valid_mask], pred))
    print('P@R>=0.7 по фолдам:', np.round(scores, 4), 'среднее:', round(np.mean(scores), 4), '| ROC-AUC:', round(np.mean(aucs), 4))
    return np.mean(scores)


def make_rf():
    return RandomForestClassifier(n_estimators=300, min_samples_leaf=3, random_state=RANDOM_STATE, n_jobs=-1)


def make_lgb():
    return lgb.LGBMClassifier(**lgb_params)

In [17]:
results = {} #добавляем признаки по шагам, чтобы видеть вклад каждой группы по сравнению с baseline

print('baseline (как в quickstart): n_events + n_items, RandomForest')
results['baseline_rf'] = run_cv(make_rf, ['n_events', 'n_items'], False)


behaviour_cols = [c for c in features.columns if c not in ['item_pop_mean', 'item_pop_max']]

print('поведенческие признаки, RandomForest')
results['behaviour_rf'] = run_cv(make_rf, behaviour_cols, False)

print('поведенческие признаки, LightGBM')
results['behaviour_lgb'] = run_cv(make_lgb, behaviour_cols, False)

print('+ популярность объявлений, LightGBM')
results['popularity_lgb'] = run_cv(make_lgb, list(features.columns), False)

print('+ доля ботов среди прошлых посетителей объявлений, LightGBM')
results['bot_rate_lgb'] = run_cv(make_lgb, list(features.columns), True)

baseline (как в quickstart): n_events + n_items, RandomForest
P@R>=0.7 по фолдам: [0.0934 0.0987 0.0931] среднее: 0.0951 | ROC-AUC: 0.6046
поведенческие признаки, RandomForest
P@R>=0.7 по фолдам: [0.6769 0.6459 0.729 ] среднее: 0.684 | ROC-AUC: 0.9199
поведенческие признаки, LightGBM
P@R>=0.7 по фолдам: [0.7733 0.7418 0.7483] среднее: 0.7545 | ROC-AUC: 0.9363
+ популярность объявлений, LightGBM
P@R>=0.7 по фолдам: [0.821  0.8333 0.8201] среднее: 0.8248 | ROC-AUC: 0.944
+ доля ботов среди прошлых посетителей объявлений, LightGBM
P@R>=0.7 по фолдам: [0.9103 0.9079 0.8898] среднее: 0.9027 | ROC-AUC: 0.9452


In [18]:
print(pd.Series(results).round(4))

baseline_rf       0.0951
behaviour_rf      0.6840
behaviour_lgb     0.7545
popularity_lgb    0.8248
bot_rate_lgb      0.9027
dtype: float64


In [19]:
#для теста доля ботов считается по всем меткам train, тест целиком идёт после train
all_rates = item_bot_rate(items, train[['cookie_id', 'target']], train['day'].max() + 1, 4)
full = features.join(all_rates)

X_full_train = full.loc[train['cookie_id']]
X_test = full.loc[test['cookie_id']]

final_model = make_lgb()
final_model.fit(X_full_train, y_train)

importance = pd.Series(final_model.feature_importances_, index=X_full_train.columns) #смотрим, какие признаки модель использует чаще всего
print(importance.sort_values(ascending=False).head(20))

item_bot_rate_mean          646
dt_q75                      609
dt_median                   592
items_ratio                 554
page_mean                   500
dt_q25                      479
n_categories                460
n_locations                 436
dt_entropy                  415
x_std                       393
cookie_age_days             383
view_dt_median              379
dt_min                      375
x_max                       373
y_max                       363
events_per_session          346
item_pop_mean               326
dt_cv                       317
share_contact_phone_show    258
y_mean                      255
dtype: int32


In [20]:
submission = pd.DataFrame()
submission['cookie_id'] = test['cookie_id'].values
submission['score'] = final_model.predict_proba(X_test)[:, 1]
#проверяем формат из условия: одна строка на каждую куку из test и score от 0 до 1
assert len(submission) == len(test)
assert submission['cookie_id'].is_unique
assert set(submission['cookie_id']) == set(test['cookie_id'])
assert submission['score'].between(0, 1).all()

submission.to_csv('submission.csv', index=False)
print(submission.head())
print(submission['score'].describe())

             cookie_id     score
0  ck_315fb710a0e371e7  0.001495
1  ck_a76ee3b3e3e522fd  0.056389
2  ck_94c9a4d382689e82  0.063521
3  ck_8eaf9509ad9462a0  0.001298
4  ck_9a88a5a989cb5bc6  0.065538
count    4909.000000
mean        0.087871
std         0.228610
min         0.000471
25%         0.004007
50%         0.010124
75%         0.027786
max         0.996819
Name: score, dtype: float64
